In [3]:
import pandas as pd
import glob
import os

# Folder containing xls files
files = glob.glob("region_wise_reports/*.xls")

all_data = []

for file in files:

    # Skip first 5 rows
    df = pd.read_excel(
        file,
        skiprows=12
    )

    # Remove fully empty rows
    df = df.dropna(how='all')

    # Remove unnamed columns
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

    # Add source filename
    df["source_file"] = os.path.basename(file)

    all_data.append(df)

# Combine all files
final_df = pd.concat(all_data, ignore_index=True)

print(final_df.head())

  NORTHERN  REGION                source_file
0          THERMAL  18_col_act-2_2022-APR.xls
1          NUCLEAR  18_col_act-2_2022-APR.xls
2            HYDRO  18_col_act-2_2022-APR.xls
3            Total  18_col_act-2_2022-APR.xls
4  WESTERN  REGION  18_col_act-2_2022-APR.xls


In [4]:
import pandas as pd
import glob

files = glob.glob("region_wise_reports/*.xls")

all_rows = []

for file in files:

    df = pd.read_excel(
        file,
        skiprows=10,
        header=None
    )

    current_region = None

    for _, row in df.iterrows():

        first_col = str(row[0]).strip().upper()

        # -------------------------
        # Detect REGION rows
        # -------------------------

        if "REGION" in first_col:

            current_region = first_col
            continue

        # -------------------------
        # Skip Total rows
        # -------------------------

        if "TOTAL" in first_col:
            continue

        # -------------------------
        # Keep only category rows
        # -------------------------

        if first_col in ["THERMAL", "HYDRO", "NUCLEAR"]:

            data = {
                "Region": current_region,
                "Category": first_col,

                # JAN-2025 PROGRAM
                "Program_Generation": row[3],

                # JAN-2025 ACTUAL
                "Actual_Generation": row[4],

                # PLF Actual
                "PLF_Actual": row[14]
            }

            all_rows.append(data)

final_df = pd.DataFrame(all_rows)

print(final_df.head())

             Region Category  Program_Generation  Actual_Generation PLF_Actual
0  NORTHERN  REGION  THERMAL             22566.0           25146.03   74.58152
1  NORTHERN  REGION  NUCLEAR               768.0             907.12  77.770919
2  NORTHERN  REGION    HYDRO              4805.0            5519.12           
3   WESTERN  REGION  THERMAL             44347.0           43682.94  70.195159
4   WESTERN  REGION  NUCLEAR              1245.0            1101.47  83.142361


In [7]:
import pandas as pd
import glob
import os
import re

files = glob.glob("region_wise_reports/*.xls")

all_data = []

for file in files:

    # -----------------------------
    # Extract month from filename
    # Example:
    # 18_col_act-2_2022-APR.xls
    # → 2022-APR
    # -----------------------------

    filename = os.path.basename(file)

    match = re.search(r'(\d{4}-[A-Z]{3})', filename)

    if match:
        period = match.group(1)
    else:
        period = None

    # -----------------------------
    # Read Excel
    # -----------------------------

    df = pd.read_excel(
        file,
        skiprows=12,
        header=None
    )

    current_region = None

    for _, row in df.iterrows():

        first_col = str(row[0]).strip().upper()

        # ---------------------------------
        # Skip empty rows
        # ---------------------------------

        if first_col == "NAN":
            continue

        # ---------------------------------
        # Detect REGION rows
        # ---------------------------------

        if "REGION" in first_col:

            current_region = first_col
            continue

        # ---------------------------------
        # Skip TOTAL rows
        # ---------------------------------

        if "TOTAL" in first_col:
            continue

        # ---------------------------------
        # Keep only required categories
        # ---------------------------------

        if first_col in ["THERMAL", "HYDRO", "NUCLEAR"]:

            data = {
                "Period": period,
                "Region": current_region,
                "Category": first_col,

                # Column D
                "Program_Generation": row[3],

                # Column E
                "Actual_Generation": row[4],

                # Column N
                "PLF_Program": row[13],

                # Column O
                "PLF_Actual": row[14]
            }

            all_data.append(data)

# ---------------------------------
# Final DataFrame
# ---------------------------------

final_df = pd.DataFrame(all_data)
print(final_df.head())

# Save clean dataset
final_df.to_csv("clean_power_data.csv", index=False)

print("Clean dataset saved.")

     Period            Region Category  Program_Generation  Actual_Generation  \
0  2022-APR  NORTHERN  REGION  THERMAL             22566.0           25146.03   
1  2022-APR  NORTHERN  REGION  NUCLEAR               768.0             907.12   
2  2022-APR  NORTHERN  REGION    HYDRO              4805.0            5519.12   
3  2022-APR   WESTERN  REGION  THERMAL             44347.0           43682.94   
4  2022-APR   WESTERN  REGION  NUCLEAR              1245.0            1101.47   

  PLF_Program PLF_Actual  
0   62.553543   74.58152  
1   70.175439  77.770919  
2                         
3   69.511826  70.195159  
4   68.077428  83.142361  
Clean dataset saved.


In [8]:
import pandas as pd
import glob
import os
import re

files = glob.glob("region_wise_reports/*.xls")

all_data = []

for file in files:

    # --------------------------------
    # Extract period from filename
    # Example:
    # 18_col_act-2_2022-APR.xls
    # -> 2022-APR
    # --------------------------------

    filename = os.path.basename(file)

    match = re.search(r'(\d{4}-[A-Z]{3})', filename)

    if match:
        period = match.group(1)
    else:
        period = None

    # --------------------------------
    # Read Excel
    # --------------------------------

    df = pd.read_excel(
        file,
        skiprows=12,
        header=None
    )

    current_region = None

    for _, row in df.iterrows():

        first_col = str(row[0]).strip().upper()

        # -----------------------------
        # Skip empty rows
        # -----------------------------

        if first_col == "NAN":
            continue

        # -----------------------------
        # Detect region rows
        # -----------------------------

        if "REGION" in first_col:

            current_region = first_col
            continue

        # -----------------------------
        # Skip total rows
        # -----------------------------

        if "TOTAL" in first_col:
            continue

        # -----------------------------
        # Keep category rows only
        # -----------------------------

        if first_col in ["THERMAL", "HYDRO", "NUCLEAR"]:

            data = {

                "Period": period,
                "Region": current_region,
                "Category": first_col,

                # B
                "Monitored_Capacity": row[1],

                # C
                "Target": row[2],

                # D
                "Program_Generation": row[3],

                # E
                "Actual_Generation": row[4],

                # G
                "Percent_Last_Year": row[6],

                # H
                "Last_Year_Generation": row[7],

                # N
                "PLF_Program": row[13],

                # O
                "PLF_Actual": row[14]
            }

            all_data.append(data)

# --------------------------------
# Final dataframe
# --------------------------------

final_df = pd.DataFrame(all_data)

# Optional cleanup
final_df = final_df.dropna(subset=["Actual_Generation"])

# Save cleaned dataset
final_df.to_csv("clean_power_data.csv", index=False)

print(final_df.head())

print("Dataset cleaned successfully.")

     Period            Region Category  Monitored_Capacity    Target  \
0  2022-APR  NORTHERN  REGION  THERMAL            51660.26  273550.0   
1  2022-APR  NORTHERN  REGION  NUCLEAR             1620.00    9348.0   
2  2022-APR  NORTHERN  REGION    HYDRO            19576.27   76243.0   
3  2022-APR   WESTERN  REGION  THERMAL            96392.49  502787.0   
4  2022-APR   WESTERN  REGION  NUCLEAR             1840.00   14160.0   

   Program_Generation  Actual_Generation  Percent_Last_Year  \
0             22566.0           25146.03         111.433262   
1               768.0             907.12         118.114583   
2              4805.0            5519.12         114.862019   
3             44347.0           43682.94          98.502586   
4              1245.0            1101.47          88.471486   

   Last_Year_Generation PLF_Program PLF_Actual  
0            127.081240   62.553543   74.58152  
1             91.208172   70.175439  77.770919  
2            159.299479                  